# POLSCI 3

## Advanced Exercise 1

**What you'll practice:** quantifying uncertainty. In lecture you learned that an estimate can be wrong for two very different reasons: **bias** and **noise**. Today you'll use real WNBA games to see the difference.

- **Part 1 (everyone):** uses only tools you've already seen in this class.
- **Part 2 (optional):** teaches new R code that goes beyond what we cover in class. Only start it if you finish Part 1.

**Heads up:** when you run `difference_in_means()` you'll see some extra columns (`Pr(>|t|)`, `CI Lower`, `CI Upper`, `DF`). We'll learn those in upcoming lectures, so ignore them for today. Focus on **Estimate**, **Std. Error**, and **t value**.

For written questions, double-click the answer cell, type your answer, and press Shift + Enter.

---
---
# PART 2 (Optional): Going beyond the class

**Only start this if you've finished Part 1.** These sections teach R tools that we *won't* cover later in the course, but that are very useful if you keep doing data analysis. Each section has an example you can run, followed by a "Your turn."

## 2.1: Creating new variables and summarizing groups

So far you've used `subset()` + `mean()` to compute group averages one group at a time. R has shortcuts.

**`ifelse(test, value if TRUE, value if FALSE)`** creates a new variable from a condition. **`abs()`** takes an absolute value.

In [ ]:
wnba$margin <- wnba$points - wnba$opp_points            # positive = the team won
wnba$blowout <- ifelse(abs(wnba$margin) >= 20, 1, 0)     # 1 if the game was decided by 20+ points

mean(wnba$blowout)   # share of team-games that were blowouts

**`aggregate(outcome ~ group, data = ..., FUN = mean)`** computes a mean for *every* group at once. You can group by more than one variable with `+`.

In [ ]:
aggregate(points ~ home_away, data = wnba, FUN = mean)

aggregate(points ~ home_away + season, data = wnba, FUN = mean)

**`order()`** sorts. `data[rows, ]` picks rows, so `data[order(data$x), ]` sorts a whole data frame by `x` (add `decreasing = TRUE` to go high to low).

In [ ]:
team.win <- aggregate(win ~ team, data = wnba, FUN = mean)
team.win[order(team.win$win, decreasing = TRUE), ]

**Your turn (2.1).**

**(a)** Use `aggregate()` to compute the average `threes_made` for each `season`. Has the three-point shot become more common?

**(b)** Use `aggregate()` and `order()` to list teams from the most to the fewest average `threes_made` per game.

In [ ]:
# (a)


# (b)


## 2.2: Writing your own function, `sapply()`, and plotting with ggplot

In Part 1 we estimated the home-court gap for 2023 and for all seasons pooled. How much does the estimate **bounce around from season to season**? Instead of copy-pasting code four times, we can write a **function** and apply it to each season with **`sapply()`**.

A function takes an input (here, a season `s`) and returns an output (here, the estimate and its standard error).

In [ ]:
home.effect.in.season <- function(s) {
    d <- subset(wnba, season == s)
    res <- difference_in_means(points ~ home_away, data = d,
                               condition1 = 'away', condition2 = 'home')
    return(c(season = s,
             estimate = as.numeric(res$coefficients),
             se = as.numeric(res$std.error)))
}

home.effect.in.season(2023)   # try it on one season first

In [ ]:
# sapply() runs the function on each season; t() flips the result so each season is a row
season.results <- as.data.frame(t(sapply(2022:2025, home.effect.in.season)))
season.results

Now a plot. In **ggplot**, you start with `ggplot(data, aes(x = ..., y = ...))` and **add layers** with `+`.

In [ ]:
ggplot(season.results, aes(x = season, y = estimate)) +
    geom_point(size = 4) +
    geom_hline(yintercept = 0, linetype = 'dashed') +
    scale_x_continuous(breaks = 2022:2025) +
    labs(title = 'Home-court advantage by season',
         x = 'Season', y = 'Home minus away points')

**Your turn (2.2).**

**(a)** Add a column `t.stat` to `season.results` (estimate divided by se). Then add a column `clears.cutoff` that is `TRUE` when `abs(t.stat) > 1.96`. Which seasons clear the cutoff on their own?

**(b)** Copy the plot code and add a red horizontal line at the pooled estimate from `dim.all` with `geom_hline(yintercept = as.numeric(dim.all$coefficients), color = 'red')`.

**(c)** In a sentence or two: the true home-court advantage presumably didn't change much from year to year. Why do the season-by-season estimates differ so much anyway?

In [ ]:
# (a)


# (b)


**(c) Your answer:** *(double-click here to type your answer)*

## 2.3: How much data do we need? A simulation

Earlier you saw that one season (t of about 1.5) couldn't clearly detect the home-court gap but four seasons could. **How many games would it take?** We can answer this by simulation, using the real data as our "world."

The idea: repeatedly draw a fake "dataset" of $n$ team-games **with replacement** from our real data, compute the t-statistic, and record how often it clears 1.96. The share of fake datasets that clear the cutoff is called **statistical power**.

New code:
- `nrow(data)`: number of rows
- `sample(1:nrow(data), n, replace = TRUE)`: pick $n$ random row numbers (the same row can be picked twice)
- `data[rows, ]`: keep those rows
- `mean()` of `TRUE/FALSE` values gives the **share** that are `TRUE`

In [ ]:
simulate.t <- function(n) {
    rows <- sample(1:nrow(wnba), n, replace = TRUE)   # draw n team-games
    fake <- wnba[rows, ]                              # build the fake dataset
    res <- difference_in_means(points ~ home_away, data = fake,
                               condition1 = 'away', condition2 = 'home')
    return(as.numeric(res$coefficients / res$std.error))
}

set.seed(10)
simulate.t(500)   # one fake dataset with 500 team-games (about one season)

In [ ]:
# Run it 300 times at each sample size and record how often |t| > 1.96
sample.sizes <- c(100, 250, 500, 1000, 2000, 4000)

power <- sapply(sample.sizes, function(n) mean(abs(replicate(300, simulate.t(n))) > 1.96))

power.results <- data.frame(n = sample.sizes, share.clearing.cutoff = power)
power.results

In [ ]:
ggplot(power.results, aes(x = n, y = share.clearing.cutoff)) +
    geom_line() +
    geom_point(size = 3) +
    ylim(0, 1) +
    labs(title = 'How often would we detect the home-court advantage?',
         x = 'Number of team-games in the dataset',
         y = 'Share of fake datasets with |t| > 1.96')

**Your turn (2.3).**

**(a)** Roughly how many team-games do we need before we detect the gap in about 80% of fake datasets? One WNBA regular season has roughly 500 team-games. About how many seasons is that?

**(b)** Why is it *not* a problem for a scientist that a small study often fails to clear the cutoff even when the effect is real? What is the problem? (Think about what you'd conclude from a single small study.)

**(c) Challenge:** What if there were **no** home-court advantage at all? Copy `simulate.t` into a new function `simulate.t.null` that first **shuffles** the home/away labels (add `fake$home_away <- sample(fake$home_away)` after building `fake`). Run it 500 times at `n = 500`. What share of fake datasets clear 1.96 even though the true effect is zero? Does the answer change with `n`?

In [ ]:
# (a)


# (c) Challenge


**(a), (b) Your answers:** *(double-click here to type your answers)*

---
## What you picked up in Part 2

- `ifelse()`, `abs()`: create new variables from conditions
- `aggregate()`, `order()`, `data[rows, ]`: summarize and sort groups
- `function()`, `return()`, `sapply()`: write reusable code and apply it repeatedly
- `ggplot()` with `geom_point()`, `geom_line()`, `geom_hline()`, `labs()`: build plots layer by layer
- `sample(..., replace = TRUE)`, `nrow()`: simulate new datasets from your data